# Melhorias Futuras no Agente RAG Financeiro
Este notebook documenta possíveis abordagens evoluir a arquitetura do RAG, focando em aprimorar a qualidade do contexto recuperado e introduzindo uma camada de avaliação automatizada.

## 1. Reranking (Cross-Encoder)
O modelo de embedding que usamos para a busca inicial (Bi-Encoder) é rápido, pois os vetores são pré-computados, mas foca mais em similaridade léxica ampla do que no grau de relevância profunda.

**Por que aplicar Reranking?**
Podemos recuperar um número maior de chunks inicialmente (ex: 15) e usar um modelo Cross-Encoder. O Cross-Encoder avalia simultaneamente o par (Query, Chunk) e retorna um score de relevância semântica altíssimo, reordenando-os para que o LLM só receba o topo 5. Isso aumenta drasticamente o nDCG do pipeline final.

In [ ]:
# Exemplo conceitual de como integrar um Cross-Encoder (ex: BGE-Reranker ou similar via LangChain)
'''
from sentence_transformers import CrossEncoder
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

# Inicializa o reranker
model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-base")
compressor = CrossEncoderReranker(model=model, top_n=5)

# Envolve o retriever original
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=retriever # Seu retriever atual do ChromaDB (trazendo mais resultados, ex: 15)
)

# Agora a busca passa por 2 estágios: Vetorial Rápida -> Reordenação Profunda
documentos_refinados = compression_retriever.invoke("What is Apple's FY2022 net income?")
'''

## 2. LLM-as-a-judge
Em vez de apenas avaliar métricas léxicas e de posição (MRR, nDCG), podemos usar um modelo secundário e leve para inspecionar a resposta gerada de forma qualitativa.
Como veremos a seguir para o llm-chunk-splitter, o processo foi custoso computacionalmente e apresentou um trade-off negativo na qualidade e perfomance

LLm como métrica binária (1=Passou, 0=Falhou) em três eixos:
- **Factual Accuracy (Acurácia):** A resposta contém números exatos provenientes do contexto ou sofreu de alucinação?
- **Completeness (Completude):** A resposta endereçou todas as partes da pergunta do usuário?
- **Relevance (Relevância):** A resposta contém jargões excessivos não solicitados ou tangenciou o tema principal?

In [ ]:
# Exemplo:
'''
EVALUATION_PROMPT = """
Você é um avaliador rigoroso. Dada a PERGUNTA do usuário, o CONTEXTO recuperado e a RESPOSTA do sistema, avalie:

1. Factual Accuracy: A RESPOSTA inventa algum número ou fato que não está no CONTEXTO? (1 = Seguro, 0 = Alucinação)
2. Completeness: A RESPOSTA atende integralmente à PERGUNTA? (1 = Sim, 0 = Incompleta)

PERGUNTA: {pergunta}
CONTEXTO: {contexto}
RESPOSTA: {resposta}

Retorne EXCLUSIVAMENTE um JSON com as chaves 'accuracy' e 'completeness'.
"""
'''

## 3. LLM como Chunk Splitter
A implementação de rodar um modelo menor para otimizar o processo de chunking pode ser utilizado em maior escala, porém, foi relativamente custoso computacionalmente o que mostrou um trade-off não válido para o meu contexto atual.

In [ ]:
#Função Exemplo
'''
def make_prompt(document):
    how_many = (len(document["text"]) // AVERAGE_CHUNK_SIZE) + 1
    doc_text = document["text"][:LLM_MAX_INPUT_CHARS]
    return f"""
You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from a public repository called financebench from PatronusAI, displaying financial information about companies.
The document is of type: {document["type"]}
The document has been retrieved from: {document["source"]}

A chatbot will use these chunks to answer finance questions about the companies.
You should divide up the document as you see fit, being sure that the entire document is returned across the chunks - don't leave anything out.
This document should probably be split into at least {how_many} chunks, but you can have more or less as appropriate, ensuring that there are individual chunks to answer specific questions.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:

{doc_text}

Return ONLY valid JSON (no markdown fences) in this exact format:
{{"chunks": [{{"headline": "...", "summary": "...", "original_text": "..."}}]}}
"""

'''